# PyHEOM v1.0: 2-Level System Population Dynamics (GPU)

> **Before running:** select *Runtime > Change runtime type* and set
> *Hardware accelerator* to **GPU** (T4 is sufficient).

This notebook simulates the population dynamics of a two-level open quantum system
using the HEOM and Redfield master equation solvers from
**pyheom v1.0** with the CUDA (GPU) backend.

**Physical parameters**
- Site energies: epsilon_1 = 100 cm^-1, epsilon_2 = 0 cm^-1
- Coupling: J = 100 cm^-1
- Bath (Drude): lambda = 20 cm^-1, gamma = 53.08 cm^-1, T = 300 K
- HEOM hierarchy depth: n_tiers = 20

Parameters follow Ishizaki & Fleming, J. Chem. Phys. **130**, 234111 (2009).

**Note**: for this small system the GPU backend is not faster than the CPU backend.
GPU acceleration becomes significant for larger systems (n_level >= 10, n_tiers >= 30).


In [ ]:
# --- Installation -- choose ONE option ----------------------------------------
import os, subprocess, sys, glob

# Option A: source build from GitHub (~15 min on Colab)
# CUDA_ARCH_LIST: T4=75, V100=70, A100=80, L4=89
env = os.environ.copy()
env['CMAKE_ARGS'] = (
    '-DLIBHEOM_ENABLE_CUDA=ON '
    '-DLIBHEOM_ENABLE_EIGEN=ON '
    '-DCUDA_ARCH_LIST=75'
)
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
     'git+https://github.com/tatsushi-ikeda/pyheom.git@master'],
    env=env,
)

# Option B: CUDA 12.x wheel from Google Drive (uncomment after CUDA wheel is available)
# 1. Comment out the subprocess.check_call block in Option A above.
# 2. Upload the cuda .whl file to My Drive on Google Drive.
# 3. Uncomment and run this block.
# from google.colab import drive
# drive.mount('/content/drive')
# wheels = sorted(glob.glob('/content/drive/MyDrive/**/*pyheom*cuda12*.whl', recursive=True))
# if not wheels:
#     raise FileNotFoundError('No pyheom CUDA 12 wheel found in Google Drive.')
# print('Installing:', wheels[-1])
# subprocess.check_call([sys.executable, '-m', 'pip', 'install',
#                        '--no-deps', '--no-cache-dir', wheels[-1]])

# Option C: GitHub Releases (uncomment after public release)
# !pip install https://github.com/tatsushi-ikeda/pyheom/releases/download/v1.0.0b1/pyheom-1.0.0b1+cuda12x-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl

# Restart runtime to apply installation and avoid stale module state.
# After restart, skip this cell and run from the next cell onward.
print('Installation complete. Restarting runtime ...')
os.kill(os.getpid(), 9)


In [ ]:
import importlib.metadata
import pyheom.pylibheom as lb

print('pyheom version :', importlib.metadata.version('pyheom'))
print('eigen backend  :', lb.eigen_is_supported())
print('mkl backend    :', lb.mkl_is_supported())
print('cuda backend   :', lb.cuda_is_supported())

if not lb.cuda_is_supported():
    raise RuntimeError(
        'CUDA backend not available. '
        'Check that the runtime type is set to GPU and that the CUDA build succeeded.'
    )


## Model

Each site is independently coupled to a Markovian (Drude) bath.
The system Hamiltonian is

$$H_S = \begin{pmatrix} \varepsilon_1 + \lambda & J \\ J & \varepsilon_2 + \lambda \end{pmatrix}$$

where the diagonal shift $\lambda$ is the bath reorganization energy.

The Drude spectral density $J(\omega) = \eta\,\gamma_c^2\,\omega/(\omega^2+\gamma_c^2)$
is decomposed into one exponential mode with `noise_decomposition(J, T, type_ltc='none')`.
The system-bath coupling operators are the site projectors $V_1 = |1\rangle\langle 1|$
and $V_2 = |2\rangle\langle 2|$.


In [ ]:
import numpy as np
from pyheom import HEOMSolver, RedfieldSolver, Drude, noise_decomposition, unit

# --- physical parameters (all in cm^-1 / fs unless noted) --------------------
epsilon_1 = 100.0        # site 1 energy  [cm^-1]
epsilon_2 =   0.0        # site 2 energy  [cm^-1]
J_12      = 100.0        # electronic coupling  [cm^-1]
lambda_c  =  20.0        # reorganization energy (= J_12 / 5)  [cm^-1]
gamma_c   =  53.08       # bath relaxation rate  [cm^-1]
T         = 208.51       # temperature ~ 300 K expressed in cm^-1 (= k_B T / hc)
n_tiers   =  20          # HEOM hierarchy depth

units = {'energy': unit.wavenumber, 'time': unit.femtosecond}

dtype = np.complex128

# --- Hamiltonian (includes diagonal reorganization energy shift) ---------------
H = np.array([[epsilon_1 + lambda_c, J_12              ],
              [J_12,                 epsilon_2 + lambda_c]], dtype=dtype)

# --- site projectors (system-bath coupling operators) -------------------------
V_1 = np.array([[1, 0], [0, 0]], dtype=dtype)   # projector onto site 1
V_2 = np.array([[0, 0], [0, 1]], dtype=dtype)   # projector onto site 2

# --- bath correlation decomposition ------------------------------------------
J = Drude(eta=2*lambda_c/gamma_c, gamma_c=gamma_c)

corr_1 = noise_decomposition(J, T=T, type_ltc='none')
corr_1.V = V_1

corr_2 = noise_decomposition(J, T=T, type_ltc='none')
corr_2.V = V_2

# --- time grid ----------------------------------------------------------------
t_list = np.arange(0.0, 1000.0, 1.0)   # 0-999 fs, 1 fs steps
dt     = 5e-2                           # ODE integration step [fs]

# --- initial state: site 1 fully excited --------------------------------------
rho_0 = np.zeros((2, 2), dtype=dtype)
rho_0[0, 0] = 1.0


In [ ]:
from tqdm.auto import tqdm
import time

qme_heom = HEOMSolver(
    H, [corr_1, corr_2],
    space='ado', format='sparse', engine='cuda', device=0,
    liouville_order='C', solver='lsrk4',
    n_tiers=n_tiers,
    units=units,
)

with tqdm(total=len(t_list), desc='HEOM') as bar:
    def _cb_heom(t):
        bar.update(1)
    t0 = time.perf_counter()
    result_heom = qme_heom.solve(
        rho_0, t_list, dt=dt,
        e_ops=[V_1, V_2],
        callback=_cb_heom,
    )
    print(f'HEOM elapsed: {time.perf_counter()-t0:.1f} s')

# result_heom.expect[i] is a complex array of shape (len(t_list),)
# Tr(V_k @ rho(t)) = rho_kk(t) = population of site k
pop_1_heom = result_heom.expect[0].real
pop_2_heom = result_heom.expect[1].real

print(f'Initial population: site 1 = {pop_1_heom[0]:.4f}, site 2 = {pop_2_heom[0]:.4f}')
print(f'Final  population: site 1 = {pop_1_heom[-1]:.4f}, site 2 = {pop_2_heom[-1]:.4f}')


In [ ]:
qme_rf = RedfieldSolver(
    H, [corr_1, corr_2],
    space='liouville', format='sparse', engine='cuda', device=0,
    liouville_order='C', solver='lsrk4',
    units=units,
)

t0 = time.perf_counter()
result_rf = qme_rf.solve(
    rho_0, t_list, dt=dt,
    e_ops=[V_1, V_2],
)
print(f'Redfield elapsed: {time.perf_counter()-t0:.1f} s')

pop_1_rf = result_rf.expect[0].real
pop_2_rf = result_rf.expect[1].real


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(result_heom.times, pop_1_heom, label='HEOM site 1', color='tab:blue')
ax.plot(result_heom.times, pop_2_heom, label='HEOM site 2', color='tab:orange')
ax.plot(result_rf.times, pop_1_rf, '--', label='Redfield site 1', color='tab:blue',   alpha=0.6)
ax.plot(result_rf.times, pop_2_rf, '--', label='Redfield site 2', color='tab:orange', alpha=0.6)
ax.set_xlabel('Time / fs')
ax.set_ylabel('Population')
ax.set_title('2-level system population dynamics (GPU)')
ax.legend()
fig.tight_layout()
plt.show()
